In [1]:
# 1. Wir installieren die Bibliothek 'pypdf', um PDFs verarbeiten zu können
!pip install pypdf

print("Erfolgreich installiert! Wir sind bereit für das PDF.")

Erfolgreich installiert! Wir sind bereit für das PDF.


In [2]:
from google.colab import files
import pypdf

# 1. Upload-Dialog starten
print("Bitte lade jetzt dein Pflege-PDF hoch:")
uploaded = files.upload()

# 2. Den Dateinamen der hochgeladenen Datei herausfinden
dateiname = list(uploaded.keys())[0]
print(f"\nDatei '{dateiname}' erfolgreich hochgeladen! Lese Text aus...")

# 3. Das PDF öffnen und den Text extrahieren
reader = pypdf.PdfReader(dateiname)
gesamter_text = ""

for seite in reader.pages:
    gesamter_text += seite.extract_text() + "\n"

print(f"\nErfolg! Das Dokument hat {len(reader.pages)} Seiten.")
print("Hier sind die ersten 500 Zeichen aus deinem PDF:")
print("-" * 40)
print(gesamter_text[:500])

Bitte lade jetzt dein Pflege-PDF hoch:


Saving handbuch-pflegedokumentation-data.pdf to handbuch-pflegedokumentation-data.pdf

Datei 'handbuch-pflegedokumentation-data.pdf' erfolgreich hochgeladen! Lese Text aus...

Erfolg! Das Dokument hat 184 Seiten.
Hier sind die ersten 500 Zeichen aus deinem PDF:
----------------------------------------
P f l e g e
Pflegedokumentation  
     stationär  
Handbuch
stationär
Information
Das Handbuch für die Pflegeleitung
Beratender Arbeitskreis und  
Testleserinnen/Testleser:
Birgit Alt-Resch
Heimleiterin Alten- und Pflegeheim Stift 
St. Irminen
Qualitätsbeauftragte des T rägers
Vereinigte Hospitien, Stiftung d. ö. R.
Irminenfreihof 2
54290 T rier
T el.: 06 51/45 12 87
E-Mail: b.alt-resch@ 
vereinigtehospitien.de
Uwe Brucker
Fachgebietsleiter Pflegerische 
Versorgung
Medizinischer Dienst der 
Spit


In [3]:
# 1. Wir installieren das Industriestandard-Tool LangChain für das Text-Splitting
!pip install langchain-text-splitters

from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. Wir definieren, wie der Text zerschnitten werden soll
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # Jeder Block soll ca. 1000 Zeichen lang sein
    chunk_overlap=200,    # 200 Zeichen Überlappung, damit wir keine Sätze in der Mitte durchschneiden
    length_function=len
)

# 3. Wir zerschneiden deinen gesamten PDF-Text
text_chunks = text_splitter.split_text(gesamter_text)

print(f"\nErfolg! Das 184-seitige PDF wurde in {len(text_chunks)} handliche Text-Blöcke aufgeteilt.")
print("Hier ist ein zufälliger Beispiel-Block (Block Nr. 15), den unsere KI später lesen kann:")
print("-" * 40)
print(text_chunks[15])


Erfolg! Das 184-seitige PDF wurde in 589 handliche Text-Blöcke aufgeteilt.
Hier ist ein zufälliger Beispiel-Block (Block Nr. 15), den unsere KI später lesen kann:
----------------------------------------
Überdokumentation waren rasch ausgemacht: Rechtsvorschriften, Rechtsspre-
chung, die Prüfinstanzen MDK und Heimaufsicht, die Anbieter von Pflegedokumen-
tationssystemen. Diese Einschätzungen zu überprüfen war ein Teil des Arbeitsauftra-
ges des BMFSFJ. Das Ergebnis lautet: Es gibt keine Rechtsvorschriften, die unnötigen 
Dokumentationsaufwand vorgeben. Aus ha ftungsrechtlichen Gründen allein muss 
nichts dokumentiert werden, was nicht auch aus pflegefachlicher Sicht dokumen-
tiert werden müsste. Die Einschätzungen, Empfehlungen und Auflagen in den Prüf-
berichten von MDK und Heimaufsicht erweis en sich weit überwiegend als fachlich 
angemessen. Und Anbieter von Pflegedokumentationssystemen reagieren mit einer 
breiten Produktpalette nicht zuletzt auf die Anforderungen ihrer Kund(en)/-

In [2]:
import os
import pypdf
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

dateiname = "handbuch-pflegedokumentation-data.pdf"

# 1. Prüfen, ob das PDF noch da ist (sonst neu hochladen)
if not os.path.exists(dateiname):
    print("Das PDF ist nach dem Neustart weg. Bitte nochmal hochladen:")
    from google.colab import files
    uploaded = files.upload()
    dateiname = list(uploaded.keys())[0]

# 2. PDF auslesen
print("1/3: Lese PDF aus...")
reader = pypdf.PdfReader(dateiname)
gesamter_text = ""
for seite in reader.pages:
    gesamter_text += seite.extract_text() + "\n"

# 3. Text in Chunks zerschneiden
print("2/3: Zerschneide Text in Blöcke...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, length_function=len)
text_chunks = text_splitter.split_text(gesamter_text)

# 4. Vektordatenbank erstellen
print("3/3: Erstelle Vektordatenbank (das dauert ca. 1-2 Minuten)...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vektor_datenbank = Chroma.from_texts(text_chunks, embeddings)

print("\nErfolg! Die Vektordatenbank ist fertig und gefüllt. Wir können jetzt Fragen stellen!")

1/3: Lese PDF aus...
2/3: Zerschneide Text in Blöcke...
3/3: Erstelle Vektordatenbank (das dauert ca. 1-2 Minuten)...

Erfolg! Die Vektordatenbank ist fertig und gefüllt. Wir können jetzt Fragen stellen!


In [4]:
# Wir geben jetzt alle 3 gefundenen Ergebnisse aus, nicht nur Platz 1
for i, ergebnis in enumerate(suchergebnisse):
    print(f"🏆 --- Treffer Platz {i+1} ---")
    print(ergebnis.page_content)
    print("\n")

🏆 --- Treffer Platz 1 ---
Praxis des behandelnden Arztes/der behandelnden Ärztin ausgewiesen wer-
den und wird in die Pflegedokumentation übertragen mit einem Hinweis auf 
das Original. Für die Umsetzung einer ärztliche Verordnung gilt, dass immer 
Spezielle Regeln zum Führen der Pflegedokumentation • Ärztliche Verordnungen                                       3.11 
  
Pflegedokumentation stationär • Das Handbuch für die Pflegeleitung            Seite 128
 
das Einverständnis des/der Betroffenen und/oder das des Bevollmächtigten 
bzw. des Betreuers/der Betreuerin vorliegt. Anhand des Stammblattes muss 
erkennbar sein, ob der/die Bewohner/-in in die medizinische Versorgung 
selbst einwilligen kann oder ob andere Personen stellvertretend tätig werden. 
Die Ärzte/Ärztinnen sind verpflichtet, die Medikamentengabe laufend zu kon-
trollieren und ggf. Korrekturen durchzuführen.  
Im Zusammenhang mit der ärztlichen Verordnung bestehen für die Einrich-


🏆 --- Treffer Platz 2 ---
bzw. der Ents

In [5]:
# 1. Wir packen alle gefundenen Textblöcke zu einem großen "Kontext-Text" zusammen
kontext = "\n\n---\n\n".join([ergebnis.page_content for ergebnis in suchergebnisse])

# 2. Wir bauen den Befehl (Prompt) für das Sprachmodell (z.B. ChatGPT)
prompt = f"""
Du bist ein freundlicher und kompetenter Pflege-Berater für die App 'hilfefuersenioren'.
Beantworte die Frage des Nutzers NUR basierend auf den folgenden Textblöcken.
Wenn die Antwort nicht im Text steht, erfinde nichts, sondern sage höflich, dass du es nicht genau weißt.

TEXTBLÖCKE (Dein Wissen aus dem Handbuch):
{kontext}

FRAGE DES NUTZERS:
{frage}

DEINE ANTWORT:
"""

# 3. Wir geben aus, was wir an die KI schicken würden
print("Das hier ist der Prompt, den dein System im Hintergrund an OpenAI (ChatGPT) schicken würde:\n")
print("=" * 60)
print(prompt)
print("=" * 60)


Das hier ist der Prompt, den dein System im Hintergrund an OpenAI (ChatGPT) schicken würde:


Du bist ein freundlicher und kompetenter Pflege-Berater für die App 'hilfefuersenioren'.
Beantworte die Frage des Nutzers NUR basierend auf den folgenden Textblöcken.
Wenn die Antwort nicht im Text steht, erfinde nichts, sondern sage höflich, dass du es nicht genau weißt.

TEXTBLÖCKE (Dein Wissen aus dem Handbuch):
Praxis des behandelnden Arztes/der behandelnden Ärztin ausgewiesen wer-
den und wird in die Pflegedokumentation übertragen mit einem Hinweis auf 
das Original. Für die Umsetzung einer ärztliche Verordnung gilt, dass immer 
Spezielle Regeln zum Führen der Pflegedokumentation • Ärztliche Verordnungen                                       3.11 
  
Pflegedokumentation stationär • Das Handbuch für die Pflegeleitung            Seite 128
 
das Einverständnis des/der Betroffenen und/oder das des Bevollmächtigten 
bzw. des Betreuers/der Betreuerin vorliegt. Anhand des Stammblattes muss 
erke